<a href="https://colab.research.google.com/github/las21-sfu/NLP-Week1-Text-Classification/blob/michael/dataset_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This notebook entails the cleaning process of our datasets to hone in on exactly the data we want to focus on for use with the model to test.

A lot of this is derived from IAT 461 work. The following have also been referenced:

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html

https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html

https://saturncloud.io/blog/exporting-dataframe-as-csv-file-from-google-colab-to-google-drive/

In [173]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [172]:
import pandas as pd

In [153]:
# Login using e.g. `huggingface-cli login` to access this dataset
df_base_training = pd.read_csv("hf://datasets/AmaanP314/youtube-comment-sentiment/youtube-comments-sentiment.csv")

In [67]:
# https://huggingface.co/datasets/community-datasets/tamilmixsentiment
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_tamil = pd.read_parquet("hf://datasets/community-datasets/tamilmixsentiment/" + splits["train"])

In [68]:
df_hinglish = pd.read_csv("hf://datasets/shae2977/hinglish-youtube-sentiments-dataset/yt_hinglish_comments_dataset.csv")

In [69]:
# Login using e.g. `huggingface-cli login` to access this dataset
df_korean = pd.read_json("hf://datasets/LLM-SocialMedia/Korean-YouTube-Comment-Sentiment-Dataset/anonymized_data.json")

In [154]:
df_base_training.head()

,CommentID,VideoID,VideoTitle,AuthorName,AuthorChannelID,CommentText,Sentiment,Likes,Replies,PublishedAt,CountryCode,CategoryID
0,UgyRjrEdJIPrf68uND14AaABAg,mcY4M9gjtsI,They killed my friend.#tales #movie #shorts,@OneWhoWandered,UC_-UEXaBL1dqqUPGkDll49A,Anyone know what movie this is?,Neutral,0,2,2025-01-15 00:54:55,NZ,1
1,UgxXxEIySAwnMNw8D7N4AaABAg,2vuXcw9SZbA,Man Utd conceding first penalty at home in yea...,@chiefvon3068,UCZ1LcZESjYqzaQRhjdZJFwg,The fact they're holding each other back while...,Positive,0,0,2025-01-13 23:51:46,AU,17
2,UgxB0jh2Ur41mcXr5IB4AaABAg,papg2tsoFzg,Welcome to Javascript Course,@Abdulla-ip8qr,UCWBK35w5Swy1iF5xIbEyw3A,waiting next video will be?,Neutral,1,0,2020-07-06 13:18:16,IN,27
3,UgwMOh95MfK0GuXLLrF4AaABAg,31KTdfRH6nY,Building web applications in Java with Spring ...,@finnianthehuman,UCwQ2Z03nOcMxWozBb_Cv66w,Thanks for the great video.\n\nI don't underst...,Neutral,0,1,2024-09-18 12:04:12,US,27
4,UgxJuUe5ysG8OSbABAl4AaABAg,-hV6aeyPHPA,After a new engine her car dies on her way hom...,@ryoutubeplaylistb6137,UCTTcJ0tsAKQokmHB2qVb1qQ,Good person helping good people.\nThis is how ...,Positive,3,1,2025-01-10 19:39:03,US,2


Okay, it looks like the key things to note are the Sentiment column, and CommentText. All of the models with different languages will need to have equivalents for these two columns to be usable.

In [155]:
df_base_training["Sentiment"].value_counts()

,count
Sentiment,
Negative,346075
Positive,343317
Neutral,342833


In [156]:
df_base_training_cleaned = df_base_training[["CommentText", "Sentiment"]]

In [157]:
df_base_training_cleaned.head()

,CommentText,Sentiment
0,Anyone know what movie this is?,Neutral
1,The fact they're holding each other back while...,Positive
2,waiting next video will be?,Neutral
3,Thanks for the great video.\n\nI don't underst...,Neutral
4,Good person helping good people.\nThis is how ...,Positive


Excellent, we've narrowed down that large dataset to the columns we need. All other dataset cleaning should be based on this.

# Tamil:  
https://huggingface.co/datasets/community-datasets/tamilmixsentiment/viewer?row=5

In [158]:
df_tamil.head()

,text,label
0,Trailer late ah parthavanga like podunga,0
1,Move pathutu vanthu trailer pakurvnga yaru,0
2,Puthupetai dhanush ah yarellam pathinga,0
3,"Dhanush oda character ,puthu sa erukay , mass ta",0
4,vera level ippa pesungada mokka nu thalaivaaaaaa,0


In [159]:
df_tamil["label"].value_counts()

,count
label,
0,7627
1,1448
2,1283
3,609
4,368


From https://huggingface.co/datasets/community-datasets/tamilmixsentiment/viewer?row=5 it appears that the value in label represent:
0 = Positive
1 = Negative
2 = Mixed_feelings
3 = unknown_state
4 = not-Tamil

In [160]:
df_tamil_cleaned = df_tamil.copy()



In [162]:
df_tamil_cleaned = df_tamil_cleaned.replace(0, "Positive")

In [163]:
df_tamil_cleaned = df_tamil_cleaned.replace(1, "Negative")

In [164]:
df_tamil_cleaned = df_tamil_cleaned.replace(2, "Neutral")

In [165]:
df_tamil_cleaned["label"].value_counts()

,count
label,
Positive,7627
Negative,1448
Neutral,1283
3,609
4,368


In [166]:
df_tamil_cleaned = df_tamil_cleaned[(df_tamil_cleaned['label'] != 3) & (df_tamil_cleaned['label'] != 4)]

In [167]:
df_tamil_cleaned["label"].value_counts()

,count
label,
Positive,7627
Negative,1448
Neutral,1283


In [168]:
df_tamil_cleaned = df_tamil_cleaned.rename(columns={"text": "CommentText", "label": "Sentiment"})

In [169]:
df_tamil_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10358 entries, 0 to 11332
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   CommentText  10358 non-null  object
 1   Sentiment    10358 non-null  object
dtypes: object(2)
memory usage: 242.8+ KB


# Hinglish

In [100]:
df_hinglish.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3190 entries, 0 to 3189
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   video_id   3190 non-null   object
 1   comment    3190 non-null   object
 2   likes      3190 non-null   int64 
 3   sentiment  3190 non-null   object
dtypes: int64(1), object(3)
memory usage: 99.8+ KB


In [101]:
df_hinglish["sentiment"].value_counts()

,count
sentiment,
Negative,1427
Positive,993
Neutral,770


In [102]:
df_hinglish_cleaned = df_hinglish[['comment', 'sentiment']].copy()

In [103]:
df_hinglish_cleaned = df_hinglish_cleaned.rename(columns={"comment": "CommentText", "sentiment": "Sentiment"})

In [104]:
df_hinglish_cleaned["Sentiment"].value_counts()

,count
Sentiment,
Negative,1427
Positive,993
Neutral,770


#Korean

In [144]:
df_korean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10035 entries, 0 to 10034
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cid                 10035 non-null  object 
 1   text                10035 non-null  object 
 2   time                10035 non-null  object 
 3   author              10035 non-null  object 
 4   channel             10035 non-null  object 
 5   votes               10035 non-null  int64  
 6   replies             10035 non-null  int64  
 7   heart               10035 non-null  bool   
 8   reply               10035 non-null  bool   
 9   time_parsed         10035 non-null  float64
 10  source_video_id     10035 non-null  object 
 11  source_video_title  10035 non-null  object 
 12  text_length         10035 non-null  int64  
 13  is_not_spam         10035 non-null  bool   
 14  votes_norm          10035 non-null  float64
 15  replies_norm        10035 non-null  float64
 16  hear

In [145]:
df_korean_cleaned = df_korean[['text', '검수_감정']].copy()

In [146]:
df_korean_cleaned['검수_감정'].value_counts()

,count
검수_감정,
,3561
부정,2403
긍정,1854
중립,1311
불명확,884
건너뛰기,19
긍정정,2
중립립,1


According to Google Translate:

*   부정 = Negative
*   긍정 = affirmation -> Positive
*   중립 = neutrality -> Neutral
*   불명확 = unclear -> (DROPPED)
*   건너뛰기 = Skip -> (DROPPED)
*   긍정정 = Positive -> (DROPPED)
*   중립립 = Neutral Lip -> (DROPPED)

If Google Translate is to be believed, this dataset has multiple columns that mean the same thing ("affirmation" and "Positive", "neutrality" and "Neutral Lip". However, for safety, I will choose to drop those.

In [147]:
df_korean_cleaned = df_korean_cleaned.replace("부정", "Negative")
df_korean_cleaned = df_korean_cleaned.replace("긍정", "Positive")
df_korean_cleaned = df_korean_cleaned.replace("중립", "Neutral")

df_korean_cleaned["검수_감정"].value_counts()

,count
검수_감정,
,3561
Negative,2403
Positive,1854
Neutral,1311
불명확,884
건너뛰기,19
긍정정,2
중립립,1


In [148]:
df_korean_cleaned = df_korean_cleaned.rename(columns={"text": "CommentText", "검수_감정": "Sentiment"})

In [149]:
df_korean_cleaned["Sentiment"].value_counts()

,count
Sentiment,
,3561
Negative,2403
Positive,1854
Neutral,1311
불명확,884
건너뛰기,19
긍정정,2
중립립,1


Those comments with no sentiment bug me. Let's drop them alongside the other non-positive/negative/neutral comments with a specific OR conditional.

In [150]:
df_korean_cleaned = df_korean_cleaned[(df_korean_cleaned['Sentiment'] == "Negative") | (df_korean_cleaned['Sentiment'] == "Positive")
| (df_korean_cleaned['Sentiment'] == "Neutral")]

In [151]:
df_korean_cleaned["Sentiment"].value_counts()

,count
Sentiment,
Negative,2403
Positive,1854
Neutral,1311


In [152]:
df_korean_cleaned = df_korean_cleaned.dropna()

In [130]:
df_korean_cleaned["Sentiment"].value_counts()

,count
Sentiment,
,3561
Negative,2403
Positive,1854
Neutral,1311


# Now, to export the datasets.

In [178]:
train_dir = '/content/drive/MyDrive/IAT 360 Final Project/Datasets/Train/'
test_dir = '/content/drive/MyDrive/IAT 360 Final Project/Datasets/Test/'
raw_dir = '/content/drive/MyDrive/IAT 360 Final Project/Datasets/Raw/'
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html


In [ ]:

df_base_training_cleaned.to_csv(train_dir + "df_base_training_cleaned.csv", index=False)
df_tamil_cleaned.to_csv(test_dir + "df_tamil_cleaned.csv", index=False)
df_hinglish_cleaned.to_csv(test_dir + "df_hinglish_cleaned.csv", index=False)
df_korean_cleaned.to_csv(test_dir + "df_korean_cleaned.csv", index=False)

In [179]:
df_base_training.to_csv(raw_dir + "df_base_training.csv", index=False)
df_tamil.to_csv(raw_dir + "df_tamil.csv", index=False)
df_hinglish.to_csv(raw_dir + "df_hinglish.csv", index=False)
df_korean.to_csv(raw_dir + "df_korean.csv", index=False)